In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import time
import threading
import cv2 as cv
import ipywidgets as widgets
from IPython.display import display
from pymycobot.mycobot import MyCobot

# =========================
# PATH SETUP
# =========================
BASE_DIR = "/home/jetson/jetcobot_ws/src/jetcobot_ctrl/scripts"
os.chdir(BASE_DIR)

# =========================
# OUTPUT
# =========================
output = widgets.Output()

# =========================
# ROBOT INIT
# =========================
mc = MyCobot("/dev/ttyUSB0", 1000000)
g_speed = 50

# =========================
# BUTTONS
# =========================
button_reset = widgets.Button(description='Reset', button_style='info')
button_power_on = widgets.Button(description='Power_on', button_style='success')
button_power_off = widgets.Button(description='Power_off', button_style='danger')

button_grasp = widgets.Button(description='GRASP', button_style='warning')
button_drop = widgets.Button(description='DROP', button_style='success')

# =========================
# ROBOT ACTIONS
# =========================
def reset_robot(b):
    with output:
        print("RESET")
    mc.power_on()
    time.sleep(1)
    mc.send_angles([0, 0, 0, 0, 0, -45], 50)
    mc.set_gripper_value(100, 50)

def power_on(b):
    mc.power_on()
    with output:
        print("POWER ON")

def power_off(b):
    mc.power_off()
    with output:
        print("POWER OFF")

# -------------------------
# GRASP (IMPORTANT PART)
# -------------------------
def grasp(b):
    with output:
        print("GRASP START")

    mc.power_on()
    time.sleep(1)

    # move above object (replace later with vision pose)
    mc.send_angles([0, -30, 20, 0, 0, -45], 50)
    time.sleep(2)

    # go down
    mc.send_angles([0, -10, 35, 0, 0, -45], 30)
    time.sleep(2)

    # close gripper
    mc.set_gripper_value(20, g_speed)
    time.sleep(1)

    # lift
    mc.send_angles([0, -30, 20, 0, 0, -45], 50)

    with output:
        print("GRASP DONE")

# -------------------------
# DROP / PLACE
# -------------------------
def drop(b):
    with output:
        print("DROP START")

    # move to drop zone
    mc.send_angles([90, -30, 20, 0, 0, -45], 50)
    time.sleep(2)

    # open gripper
    mc.set_gripper_value(100, g_speed)
    time.sleep(1)

    # return home
    mc.send_angles([0, 0, 0, 0, 0, -45], 50)

    with output:
        print("DROP DONE")

# =========================
# BIND BUTTONS
# =========================
button_reset.on_click(reset_robot)
button_power_on.on_click(power_on)
button_power_off.on_click(power_off)
button_grasp.on_click(grasp)
button_drop.on_click(drop)

# =========================
# SLIDERS
# =========================
def on_S1(v): mc.send_angle(1, v, g_speed)
def on_S2(v): mc.send_angle(2, v, g_speed)
def on_S3(v): mc.send_angle(3, v, g_speed)
def on_S4(v): mc.send_angle(4, v, g_speed)
def on_S5(v): mc.send_angle(5, v, g_speed)
def on_S6(v): mc.send_angle(6, v, g_speed)
def on_S7(v): mc.set_gripper_value(v, g_speed)

slider_S1 = widgets.IntSlider(description='J1', min=-168, max=168)
slider_S2 = widgets.IntSlider(description='J2', min=-135, max=90)
slider_S3 = widgets.IntSlider(description='J3', min=-150, max=150)
slider_S4 = widgets.IntSlider(description='J4', min=-145, max=145)
slider_S5 = widgets.IntSlider(description='J5', min=-165, max=165)
slider_S6 = widgets.IntSlider(description='J6', min=-180, max=180)
slider_S7 = widgets.IntSlider(description='G', min=10, max=100, value=100)

def connect(slider, func):
    slider.observe(lambda change: func(change["new"]), names="value")

connect(slider_S1, on_S1)
connect(slider_S2, on_S2)
connect(slider_S3, on_S3)
connect(slider_S4, on_S4)
connect(slider_S5, on_S5)
connect(slider_S6, on_S6)
connect(slider_S7, on_S7)

box_joints = widgets.VBox([
    slider_S7,
    slider_S6,
    slider_S5,
    slider_S4,
    slider_S3,
    slider_S2,
    slider_S1
])

# =========================
# RESET POSITION
# =========================
mc.power_on()
time.sleep(1)
mc.send_angles([0, 0, 0, 0, 0, -45], 50)
mc.set_gripper_value(100, 50)

# =========================
# CAMERA
# =========================
imgbox = widgets.Image(format='jpg', width=640, height=480)

def camera():
    cap = cv.VideoCapture(0, cv.CAP_V4L2)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        _, jpg = cv.imencode('.jpg', frame)
        imgbox.value = jpg.tobytes()

    cap.release()

threading.Thread(target=camera, daemon=True).start()

# =========================
# JOINT IMAGE
# =========================
joint_map_widget = widgets.Image(format='png', width=320, height=400)

img_path = os.path.join(BASE_DIR, "jetcobot_joints.png")

try:
    with open(img_path, "rb") as f:
        joint_map_widget.value = f.read()
    print("Joint image loaded")
except Exception as e:
    with output:
        print("Joint image not found:", e)

# =========================
# UI LAYOUT
# =========================
box_controls = widgets.VBox([
    button_reset,
    button_power_on,
    button_power_off,
    button_grasp,
    button_drop,
    output
])

box_middle = widgets.VBox([box_joints, joint_map_widget])

display(widgets.HBox([box_controls, box_middle, imgbox]))